In [1]:
import duckdb
from pathlib import Path

BRONZE = Path("../data/bronze").resolve().as_posix()
con = duckdb.connect()
con.sql(f"CREATE VIEW main AS SELECT * FROM read_parquet('{BRONZE}/main/*.parquet', union_by_name = true)")
con.sql("SELECT anio_archivo, count(*) AS procesos FROM main GROUP BY 1 ORDER BY 1")

┌──────────────┬──────────┐
│ anio_archivo │ procesos │
│    int32     │  int64   │
├──────────────┼──────────┤
│         2019 │    64301 │
│         2020 │    65097 │
│         2021 │    61296 │
│         2022 │    59922 │
│         2023 │    46116 │
│         2024 │   108451 │
│         2025 │   124538 │
│         2026 │   134917 │
└──────────────┴──────────┘

In [2]:
con.sql("""
    SELECT veces, count(*) AS procesos
    FROM (SELECT ocid, count(*) AS veces FROM main GROUP BY ocid)
    GROUP BY 1 ORDER BY 1
""")

┌───────┬──────────┐
│ veces │ procesos │
│ int64 │  int64   │
├───────┼──────────┤
│     1 │   664638 │
└───────┴──────────┘

In [3]:
con.sql("""
    SELECT substr(tender_datePublished, 1, 4) AS anio_publicacion, count(*) AS procesos
    FROM main GROUP BY 1 ORDER BY 1
""")

┌──────────────────┬──────────┐
│ anio_publicacion │ procesos │
│     varchar      │  int64   │
├──────────────────┼──────────┤
│ 2013             │      148 │
│ 2014             │    11501 │
│ 2015             │    52388 │
│ 2016             │    23311 │
│ 2017             │    10071 │
│ 2018             │      125 │
│ 2019             │    64494 │
│ 2020             │    66023 │
│ 2021             │    63802 │
│ 2022             │    79131 │
│ 2023             │    76571 │
│ 2024             │    80111 │
│ 2025             │    79105 │
│ 2026             │    57856 │
│ NULL             │        1 │
└──────────────────┴──────────┘
  15 rows           2 columns

In [4]:
con.sql("""
    SELECT count(*) AS procesos_desde_2019
    FROM main
    WHERE tender_datePublished >= '2019'
""")

┌─────────────────────┐
│ procesos_desde_2019 │
│        int64        │
├─────────────────────┤
│              567093 │
└─────────────────────┘

In [5]:
megas = sum(p.stat().st_size for p in Path(BRONZE).rglob("*.parquet")) / 1e6
print(f"{megas:,.1f} MB en Parquet")

186.9 MB en Parquet
